# D3-03 Inspecting and comparing premise outputs in Brightway

Audience:
- Participants who already generated the Paris course `premise` databases in `D3-02`.

Prerequisites:
- `paris-lca-course-2026` is active.
- The `paris-remind-eu-...` databases were written successfully.

Learning goals:
- Confirm that the generated scenario databases are present.
- Search consistent activities across scenarios and years.
- Run a compact comparison in Brightway before moving into Activity Browser Part II.


## Outline

1. List the generated scenario databases.
2. Pick a comparable electricity activity.
3. Compute a simple score table across scenarios and years.
4. Identify what to compare next in Activity Browser.


In [ ]:
import bw2calc as bc
import bw2data as bd
import pandas as pd

project_name = 'paris-lca-course-2026'
db_prefix = 'paris-remind-eu-'
method = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')

bd.projects.set_current(project_name)
scenario_dbs = sorted(name for name in bd.databases if name.startswith(db_prefix))

pd.DataFrame({'database_name': scenario_dbs})


## Step 1 - Pick a comparable electricity activity in each database

Electricity is the main emphasis in this course, so we start there.


In [ ]:
def choose_activity(database):
    preferred_terms = [
        ('market group for electricity, medium voltage', 'RER'),
        ('market for electricity, medium voltage', 'RER'),
        ('electricity, medium voltage', None),
    ]
    for term, preferred_location in preferred_terms:
        for activity in database.search(term):
            if preferred_location is None or activity.get('location') == preferred_location:
                return activity
    return database.random()

activity_table = []
chosen_activities = {}
for db_name in scenario_dbs:
    db = bd.Database(db_name)
    activity = choose_activity(db)
    chosen_activities[db_name] = activity
    activity_table.append(
        {
            'database_name': db_name,
            'activity_name': activity['name'],
            'location': activity.get('location'),
            'reference_product': activity.get('reference product'),
        }
    )

pd.DataFrame(activity_table)


## Step 2 - Run a compact score comparison

This is a quick Brightway-side check before the richer GUI comparisons in Activity Browser.


In [ ]:
rows = []
for db_name, activity in chosen_activities.items():
    lca = bc.LCA({activity: 1}, method)
    lca.lci()
    lca.lcia()
    rows.append(
        {
            'database_name': db_name,
            'activity_name': activity['name'],
            'location': activity.get('location'),
            'score': lca.score,
        }
    )

results = pd.DataFrame(rows)
results[['scenario', 'year']] = results['database_name'].str.extract(r'paris-remind-eu-(.*)-(\d{4})')
results.sort_values(['scenario', 'year'])


## Step 3 - Prepare the Activity Browser comparison tasks

Translate the table above into GUI comparison questions.


In [ ]:
ab_tasks = [
    'Compare SSP2-NPi versus SSP2-PkBudg1000 for the same year.',
    'Compare 2025, 2035, and 2050 within the same pathway.',
    'Inspect the activity graph of one electricity market to see which suppliers change.',
    'Export one plot or result table for discussion.',
]
ab_tasks


## Exercises

- Replace electricity with another sector activity and rerun the comparison.
- Compare absolute scores with relative score changes across years.
- Note one case where Activity Browser will be easier than a notebook.


In [ ]:
# Exercise answer scaffold
follow_up = {
    'second_sector': '...',
    'relative_change_observation': '...',
    'why_gui_helps': '...',
}
follow_up
